In [12]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

class Generator(BaseModel):
    name: str
    at_bus: str
    cost_0: float
    cost_1: float
    cost_2: float
    max_capacity_mw: float
    min_capacity_mw: float = 0.0

class Bus(BaseModel):
    name: str
    load_mw: float

class Line(BaseModel):
    name: str
    from_bus: str
    to_bus: str
    max_flow_mw: float
    reactance: float


class DC_OPF_Problem(BaseModel):
    objective_type: str
    generators: list[Generator]
    lines: list[Line]
    buses: list[Bus]

In [53]:
# 1. Load the hidden API key from your .env file
load_dotenv()

# 2. Initialize the Gemini client 
client = genai.Client()

# 3. Define the plain-text problem (Unstructured Data)
prompt_text = """
Hey team, we need to calculate the dispatch for the regional grid today. There are four main areas: the North Hub, the City Center, the Industrial Park, and the Sunny Suburb. Let's use the North Hub as our reference angle.

For demand, the City Center is pulling heavily today, about 450 MW. The Industrial Park needs 200 MW, and the Sunny Suburb is only pulling 50 MW. The North Hub doesn't have any local load.

Here's what we have for generation. Up at the North Hub, we have a massive coal plant. It costs $200 just to turn it on, plus $20 per megawatt, and a 0.05 quadratic penalty. It can push up to 600 MW, but because it's a thermal plant, it cannot be turned down below 100 MW. Down in the Sunny Suburb, we have a solar farm. Since it's renewable, the power is completely free—no startup cost, no linear cost, no quadratic cost. It maxes out at 100 MW. Finally, there's an emergency gas peaker right in the City Center. It's expensive: $500 flat startup, $60 a megawatt, and a 0.1 squared factor. It can generate up to 200 MW, but we can turn it off completely if we don't need it.

The transmission wires are a bit of a mess. The Main Corridor line goes from North Hub to City Center. It has a reactance of 0.01 and can carry 300 MW. The Heavy Line runs from North Hub to the Industrial Park (reactance 0.02, limit 250 MW). There is a cross-town tie connecting the Industrial Park to the City Center with a reactance of 0.02 and a 150 MW limit. To get power to the suburbs, we have the Suburban Feed running from City Center to Sunny Suburb (reactance 0.05, max 100 MW), and a rural bypass line connecting North Hub directly to Sunny Suburb (reactance 0.04, limit 50 MW).

Get me the cheapest dispatch without breaking the wires.
"""

# 4. Ask Gemini to read the text and fill out the Pydantic form
print("Sending text to Gemini...")
response = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=prompt_text,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DC_OPF_Problem,
        temperature=0.0,
    ),
)

# 5. Convert the JSON response back into our Pydantic object
extracted_data = DC_OPF_Problem.model_validate_json(response.text)

print("\n--- Extraction Success! ---")
print(f"Objective: {extracted_data.objective_type}")
for gen in extracted_data.generators:
    print(f"Found {gen.name}: Min {gen.min_capacity_mw}MW, Max {gen.max_capacity_mw}MW, Cost {gen.cost_2}*P^2 + {gen.cost_1}*P + {gen.cost_0}, at {gen.at_bus}")
for bus in extracted_data.buses:
    print(f"Found {bus.name}: laod {bus.load_mw}MW")
for l in extracted_data.lines:
    print(f"Found {l.name}: From {l.from_bus} to {l.to_bus}, Max Flow {l.max_flow_mw}MW, Reactance {l.reactance}")

Sending text to Gemini...

--- Extraction Success! ---
Objective: economic_dispatch
Found North Hub Coal: Min 100.0MW, Max 600.0MW, Cost 0.05*P^2 + 20.0*P + 200.0, at North Hub
Found Sunny Suburb Solar: Min 0.0MW, Max 100.0MW, Cost 0.0*P^2 + 0.0*P + 0.0, at Sunny Suburb
Found City Center Gas Peaker: Min 0.0MW, Max 200.0MW, Cost 0.1*P^2 + 60.0*P + 500.0, at City Center
Found North Hub: laod 0.0MW
Found City Center: laod 450.0MW
Found Industrial Park: laod 200.0MW
Found Sunny Suburb: laod 50.0MW
Found Main Corridor: From North Hub to City Center, Max Flow 300.0MW, Reactance 0.01
Found Heavy Line: From North Hub to Industrial Park, Max Flow 250.0MW, Reactance 0.02
Found Cross-town Tie: From Industrial Park to City Center, Max Flow 150.0MW, Reactance 0.02
Found Suburban Feed: From City Center to Sunny Suburb, Max Flow 100.0MW, Reactance 0.05
Found Rural Bypass: From North Hub to Sunny Suburb, Max Flow 50.0MW, Reactance 0.04


In [54]:
import pyomo.environ as pyo
model = pyo.ConcreteModel()

# extract the paramters
model.generator = pyo.Set(initialize = [g.name for g in extracted_data.generators])
model.line = pyo.Set(initialize = [l.name for l in extracted_data.lines])
model.bus = pyo.Set(initialize= [b.name for b in extracted_data.buses])

## parameters of generator
model.gen_power = pyo.Var(model.generator)

costs = {g.name: [g.cost_0, g.cost_1, g.cost_2] for g in extracted_data.generators}
model.cost = pyo.Param(model.generator, initialize=costs)
model.theta = pyo.Var(model.bus, domain=pyo.Reals)
gen_loc = {g.name: g.at_bus for g in extracted_data.generators}
model.gen_loc = pyo.Param(model.generator, initialize=gen_loc)

min_capacity = {g.name: g.min_capacity_mw for g in extracted_data.generators}
model.min_capicity = pyo.Param(model.generator, initialize=min_capacity)
max_capacity = {g.name: g.max_capacity_mw for g in extracted_data.generators}
model.max_capicity = pyo.Param(model.generator, initialize=max_capacity)

# parameters of line
from_bus = {l.name: l.from_bus for l in extracted_data.lines}
to_bus = {l.name: l.to_bus for l in extracted_data.lines}
model.line_from = pyo.Param(model.line, initialize=from_bus)
model.line_to = pyo.Param(model.line, initialize=to_bus)

susceptancee = {l.name: 1/l.reactance for l in extracted_data.lines}
model.B = pyo.Param(model.line, initialize=susceptancee)

max_flow = {l.name: l.max_flow_mw for l in extracted_data.lines}
model.max_flow = pyo.Param(model.line, initialize=max_flow)

# parameters of bus
bus_load = {b.name: b.load_mw for b in extracted_data.buses}
model.bus_load = pyo.Param(model.bus, initialize=bus_load)


# Constrainsts
def max_rule(model, g): # generator limit
    return model.gen_power[g] <= model.max_capicity[g]
def min_rule(model, g):
    return model.gen_power[g] >= model.min_capicity[g]
model.c1 = pyo.Constraint(model.generator, rule = max_rule)
model.c2 = pyo.Constraint(model.generator, rule = min_rule)

model.ref_angle = pyo.Constraint(expr= model.theta[model.bus.first()] == 0) # set the first bus as reference

def calc_line_flow(model, line):
    start_bus = model.line_from[line]
    end_bus = model.line_to[line]
    return model.B[line] * (model.theta[start_bus] - model.theta[end_bus])
model.line_flow = pyo.Expression(model.line, rule=calc_line_flow)

def line_limit_rule(model, line): # line flow limit
    return (-model.max_flow[line], model.line_flow[line], model.max_flow[line])
model.line_limits = pyo.Constraint(model.line, rule=line_limit_rule)

def Nodal_rule(model, bus): # nodal balance
    generator_mw = 0
    for g in model.generator:
        if model.gen_loc[g] == bus:
            generator_mw += model.gen_power[g]
    load_mw = model.bus_load[bus]
    flow_in = sum(model.line_flow[l] for l in model.line if model.line_to[l] == bus)
    flow_out = sum(model.line_flow[l] for l in model.line if model.line_from[l] == bus)
    return (generator_mw - load_mw) == (flow_out - flow_in)
model.nodal_balance = pyo.Constraint(model.bus, rule=Nodal_rule)

# Objective function
model.f1 = pyo.Objective(expr = sum(model.cost[g][2] * model.gen_power[g]**2 + model.cost[g][1] * model.gen_power[g] + model.cost[g][0] for g in model.generator), sense=pyo.minimize)

# Solve
solver = pyo.SolverFactory('ipopt')
results = solver.solve(model)
for g in model.generator:
    print(f"{g}: {pyo.value(model.gen_power[g])} MW")

North Hub Coal: 480.5555590831936 MW
Sunny Suburb Solar: 100.00000099996663 MW
City Center Gas Peaker: 119.44443991683981 MW
